# 04 - Métricas de preços e dispersão

As métricas principais comparam gasolina e etanol por posto, município e semana de referência, preservando os registros sinalizados no Notebook 02. Mínimos, máximos, amplitude, desvio-padrão e coeficiente de variação entre postos são calculados sobre as médias semanais dos postos.

Os rankings são separados por município, combustível e semana; portanto, não comparam diretamente preços de anos distintos. Empates recebem a mesma posição. Grupos com menos de dois postos são marcados como insuficientes para comparação entre estabelecimentos, e a dispersão não é interpretável quando há apenas um posto.

As variações temporais comparam cada preço à observação anterior disponível da mesma série. Os arquivos incluem a semana anterior e o intervalo em dias, para que lacunas de coleta não sejam confundidas com variações semanais consecutivas. Os cálculos de paridade permanecem como referência secundária.

In [5]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "Dados").exists() and (RAIZ.parent / "Dados").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "Dados").exists(), "Confira a pasta aberta no VS Code."
PASTA_ANALITICO = RAIZ / "Dados/Analitico"
PASTA_TABELAS = RAIZ / "tabelas"
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

posto = pd.read_parquet(PASTA_ANALITICO / "posto_semana.parquet")
municipio = pd.read_parquet(PASTA_ANALITICO / "municipio_semana.parquet")
precos = pd.read_parquet(PASTA_ANALITICO / "precos_municipio_semana.parquet")

chaves_periodo = ["semana", "estado", "municipio", "produto"]

# Dispersão por município, combustível e semana, calculada sobre médias posto-semana.
dispersao = (
    posto.groupby(chaves_periodo, as_index=False)
    .agg(
        preco_medio_postos=("preco_medio", "mean"),
        preco_min=("preco_medio", "min"),
        preco_max=("preco_medio", "max"),
        desvio_padrao=("preco_medio", "std"),
        postos=("id_posto", "nunique"),
        coletas=("coletas", "sum")
    )
)
dispersao["amplitude"] = dispersao["preco_max"] - dispersao["preco_min"]
dispersao["diferenca_preco"] = dispersao["amplitude"]
dispersao["cv_pct"] = 100 * dispersao["desvio_padrao"] / dispersao["preco_medio_postos"]
dispersao["comparacao_suficiente"] = dispersao["postos"] >= 2
dispersao.to_parquet(PASTA_ANALITICO / "metricas_dispersao.parquet", index=False)

# Rankings dentro do mesmo município, combustível e semana de coleta.
ranking_postos = posto.sort_values(chaves_periodo + ["preco_medio"]).copy()
grupo_ranking = ranking_postos.groupby(chaves_periodo, dropna=False)
ranking_postos["postos_no_periodo"] = grupo_ranking["id_posto"].transform("nunique")
ranking_postos["rank_mais_barato"] = ranking_postos.groupby(
    chaves_periodo, dropna=False
)["preco_medio"].rank(method="min", ascending=True)
ranking_postos["rank_mais_caro"] = ranking_postos.groupby(
    chaves_periodo, dropna=False
)["preco_medio"].rank(method="min", ascending=False)
ranking_postos["preco_min_entre_postos"] = grupo_ranking["preco_medio"].transform("min")
ranking_postos["preco_max_entre_postos"] = grupo_ranking["preco_medio"].transform("max")
ranking_postos["diferenca_preco_entre_postos"] = (
    ranking_postos["preco_max_entre_postos"] - ranking_postos["preco_min_entre_postos"]
)
ranking_postos["comparacao_suficiente"] = ranking_postos["postos_no_periodo"] >= 2
ranking_postos["observacao_comparacao"] = ""
ranking_postos.loc[
    ~ranking_postos["comparacao_suficiente"], "observacao_comparacao"
] = "Apenas um posto com dado nesta semana; sem comparação entre estabelecimentos."
ranking_postos.to_parquet(
    PASTA_ANALITICO / "ranking_postos_semana.parquet", index=False
)


def adicionar_variacao_temporal(base, grupos_serie):
    resultado = base.sort_values(grupos_serie + ["semana"]).copy()
    agrupado = resultado.groupby(grupos_serie, dropna=False)
    resultado["semana_anterior"] = agrupado["semana"].shift(1)
    resultado["preco_medio_anterior"] = agrupado["preco_medio"].shift(1)
    resultado["variacao_preco"] = (
        resultado["preco_medio"] - resultado["preco_medio_anterior"]
    )
    denominador = resultado["preco_medio_anterior"].where(
        resultado["preco_medio_anterior"] != 0
    )
    resultado["variacao_pct"] = 100 * resultado["variacao_preco"] / denominador
    resultado["intervalo_dias"] = (
        resultado["semana"] - resultado["semana_anterior"]
    ).dt.days
    resultado["comparacao_semanal_consecutiva"] = resultado["intervalo_dias"].eq(7)
    return resultado


metricas_posto = adicionar_variacao_temporal(
    posto, ["estado", "municipio", "produto", "id_posto"]
)
metricas_posto.to_parquet(
    PASTA_ANALITICO / "metricas_posto_semana.parquet", index=False
)

metricas_municipio = adicionar_variacao_temporal(
    municipio, ["estado", "municipio", "produto"]
)
metricas_municipio.to_parquet(
    PASTA_ANALITICO / "metricas_municipio_semana.parquet", index=False
)

# Resumo histórico descritivo; não é usado para ranquear postos entre períodos distintos.
media_posto = (
    posto.groupby(["municipio", "produto", "id_posto"], as_index=False)
    .agg(preco_medio_posto=("preco_medio", "mean"), semanas_observadas=("semana", "nunique"))
)
resumo_postos = (
    media_posto.groupby(["municipio", "produto"], as_index=False)
    .agg(
        preco_medio=("preco_medio_posto", "mean"),
        preco_min=("preco_medio_posto", "min"),
        preco_max=("preco_medio_posto", "max"),
        desvio_padrao=("preco_medio_posto", "std"),
        postos=("id_posto", "nunique"),
        semanas_observadas=("semanas_observadas", "sum")
    )
)
resumo_postos["amplitude"] = resumo_postos["preco_max"] - resumo_postos["preco_min"]
resumo_postos["cv_pct"] = 100 * resumo_postos["desvio_padrao"] / resumo_postos["preco_medio"]
resumo_postos.to_parquet(PASTA_ANALITICO / "resumo_postos.parquet", index=False)

# Mantém os indicadores de paridade como saída secundária para compatibilidade.
metricas = precos.dropna(subset=["ETANOL", "GASOLINA"]).copy()
metricas["razao"] = metricas["ETANOL"] / metricas["GASOLINA"]
metricas["limiar_razao"] = 0.70
metricas["vantajoso"] = metricas["razao"] <= metricas["limiar_razao"]
metricas.to_parquet(PASTA_ANALITICO / "metricas.parquet", index=False)

resumo_vantagem = (
    metricas.groupby("municipio", as_index=False)
    .agg(
        semanas_com_par=("vantajoso", "size"),
        semanas_vantajosas=("vantajoso", "sum"),
        percentual_semanas_vantajosas=("vantajoso", lambda valores: 100 * valores.mean()),
        razao_media=("razao", "mean"),
        razao_mediana=("razao", "median")
    )
)
resumo_vantagem.to_csv(
    PASTA_TABELAS / "semanas_vantajosas.csv", index=False, sep=";", decimal=","
)
resumo_postos.to_csv(
    PASTA_TABELAS / "resumo_postos.csv", index=False, sep=";", decimal=","
)

print("Postos com identificação validada:", posto["id_posto"].nunique())
print("Linhas posto-semana com métricas temporais:", len(metricas_posto))
print("Linhas município-semana com métricas temporais:", len(metricas_municipio))
print("Linhas de ranking posto-semana:", len(ranking_postos))
print("Grupos semanais com menos de dois postos:", int((~dispersao["comparacao_suficiente"]).sum()))
print("Grupos semanais com CV indefinido:", int(dispersao["cv_pct"].isna().sum()))
print("Paridade calculada em semanas com ambos os combustíveis:", len(metricas))
print("Semanas em que o etanol atingiu a regra de 0,70:", int(metricas["vantajoso"].sum()))
display(resumo_postos)
display(dispersao.head())

Postos com identificação validada: 205
Linhas posto-semana com métricas temporais: 69285
Linhas município-semana com métricas temporais: 5249
Linhas de ranking posto-semana: 69285
Grupos semanais com menos de dois postos: 7
Grupos semanais com CV indefinido: 7
Paridade calculada em semanas com ambos os combustíveis: 2624
Semanas em que o etanol atingiu a regra de 0,70: 78


,municipio,produto,preco_medio,preco_min,preco_max,desvio_padrao,postos,semanas_observadas,amplitude,cv_pct
0,BAYEUX,ETANOL,3.121524,2.096968,4.580191,0.726691,20,5099,2.483223,23.280019
1,BAYEUX,GASOLINA,4.108782,2.557143,5.957372,1.059052,20,5635,3.400229,25.775319
2,CABEDELO,ETANOL,3.111277,2.05015,4.79,0.692557,17,5540,2.73985,22.259576
3,CABEDELO,GASOLINA,3.924452,2.534,6.456,0.998234,17,5526,3.922,25.436277
4,JOAO PESSOA,ETANOL,3.05239,1.884,4.7405,0.780126,156,21755,2.8565,25.557879
5,JOAO PESSOA,GASOLINA,3.916679,2.378,6.188833,1.084983,156,22523,3.810833,27.701603
6,SANTA RITA,ETANOL,2.456508,2.042567,3.138,0.369358,12,1546,1.095433,15.035907
7,SANTA RITA,GASOLINA,2.964535,2.468874,3.745625,0.458007,12,1661,1.276751,15.449532


,semana,estado,municipio,produto,preco_medio_postos,preco_min,preco_max,desvio_padrao,postos,coletas,amplitude,diferenca_preco,cv_pct,comparacao_suficiente
0,2011-01-03,PB,BAYEUX,ETANOL,1.822222,1.78,1.94,0.066854,9,9,0.16,0.16,3.668812,True
1,2011-01-03,PB,BAYEUX,GASOLINA,2.385556,2.28,2.56,0.120531,9,9,0.28,0.28,5.052543,True
2,2011-01-03,PB,CABEDELO,ETANOL,1.875556,1.76,1.96,0.069482,9,9,0.2,0.2,3.70462,True
3,2011-01-03,PB,CABEDELO,GASOLINA,2.48,2.29,2.59,0.095131,9,9,0.3,0.3,3.835947,True
4,2011-01-03,PB,JOAO PESSOA,ETANOL,1.805583,1.689,2.05,0.090558,48,48,0.361,0.361,5.015458,True


In [6]:
print("Métricas de dispersão entre postos:")
display(dispersao.head())
print("Grupos com posto único (desvio-padrão indefinido):", int(dispersao["desvio_padrao"].isna().sum()))
print("Postos com identificação validada:", posto["id_posto"].nunique())

Métricas de dispersão entre postos:


,semana,estado,municipio,produto,preco_medio_postos,preco_min,preco_max,desvio_padrao,postos,coletas,amplitude,diferenca_preco,cv_pct,comparacao_suficiente
0,2011-01-03,PB,BAYEUX,ETANOL,1.822222,1.78,1.94,0.066854,9,9,0.16,0.16,3.668812,True
1,2011-01-03,PB,BAYEUX,GASOLINA,2.385556,2.28,2.56,0.120531,9,9,0.28,0.28,5.052543,True
2,2011-01-03,PB,CABEDELO,ETANOL,1.875556,1.76,1.96,0.069482,9,9,0.2,0.2,3.70462,True
3,2011-01-03,PB,CABEDELO,GASOLINA,2.48,2.29,2.59,0.095131,9,9,0.3,0.3,3.835947,True
4,2011-01-03,PB,JOAO PESSOA,ETANOL,1.805583,1.689,2.05,0.090558,48,48,0.361,0.361,5.015458,True


Grupos com posto único (desvio-padrão indefinido): 7
Postos com identificação validada: 205


In [7]:
ipca_path = RAIZ / "Dados/Analitico/ipca.csv"
if ipca_path.exists():
    ipca = pd.read_csv(ipca_path, sep=";")
    ipca.columns = ["data", "indice_ipca"]
    ipca["data"] = pd.to_datetime(ipca["data"], format="%Y-%m-%d", errors="coerce")
    ipca["indice_ipca"] = pd.to_numeric(ipca["indice_ipca"].str.replace(",", ".", regex=False), errors="coerce")
    ipca = ipca.dropna(subset=["data", "indice_ipca"]).sort_values("data")
    idx_base = ipca["indice_ipca"].iloc[0]
    ipca["indice_relativo"] = 100 * ipca["indice_ipca"] / idx_base
    display(ipca.head())
else:
    print("Arquivo de IPCA não encontrado. Use uma fonte oficial, registre a data de obtenção e o mês-base antes de deflacionar os preços.")

Arquivo de IPCA não encontrado. Use uma fonte oficial, registre a data de obtenção e o mês-base antes de deflacionar os preços.
